# Deep Q-Learning on Atari Space Invaders: DQN vs Double DQN

CS6482 Deep Reinforcement Learning, Assignment 2 (Adesayo Akintunde).

A Vanilla DQN and a Double DQN agent trained on `ALE/SpaceInvaders-v5` with TensorFlow/Keras and Gymnasium, starting from the official Keras DQN example (https://keras.io/examples/rl/deep_q_network_breakout/).


In [2]:
import os, time, random, collections
import numpy as np
import tensorflow as tf
import keras
from keras import layers
import gymnasium as gym
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation
import ale_py
import matplotlib.pyplot as plt
import matplotlib.animation

# Register Atari environments with Gymnasium
gym.register_envs(ale_py)

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("Gymnasium:", gym.__version__)

TensorFlow: 2.21.0
Keras: 3.14.0
Gymnasium: 1.3.0


---
## 1. Why Reinforcement Learning?

Reinforcement Learning is used for Atari games because of its decision making under moments of uncertainty. Supervised Learning requires labelled action e.g correct action, but in atari games no such thing exists, we dont know which is move is the optimal choice. Unsupervised learning can tell us what moves look alike but not the mechanism to get towards our goals.

RL directly models the interaction loop that games impose:

1. The **agent** observes a state.
2. It selects an **action** (move left, fire, etc.).
3. The **environment** transitions to a new state and emits a **reward** (score change).
4. The agent updates its policy to maximise cumulative future reward.

This closed loop — observe → act → receive reward → update — matches RL's Markov Decision
Process (MDP) formalism exactly. The temporal credit-assignment problem (which past actions
caused a reward much later?) is solved by Q-learning, which propagates future rewards
backwards through time via the Bellman equation. No other ML paradigm offers this
combination of trial-and-error exploration, delayed reward handling, and policy optimisation,
making RL the paradigm of choice for game-playing agents (Mnih et al., 2015).

**References:**
- Nussenbaum, K. and Hartley, C.A., 2019. Reinforcement learning across development: What insights can we draw from a decade of research?. Developmental cognitive neuroscience, 40, p.100733.


---
## 2. The Environment

### Game Selected: Space Invaders (ALE/SpaceInvaders-v5)

I chose Space Invaders, a classic Atari 2600 arcade game where the player controls a laser
cannon and moves horizontally along the screen avoiding getting hit by a wave of alien invaders. 
The player can shoot them to stop them from reaching the bottom of the screen.
The game is completed when all lives are lost or all aliens are eliminated.

It is a well-studied benchmark in deep RL because:
- The reward signal is delayed and sparse.
- It has a small action set (6 actions).

### Preprocessing Pipeline
The raw 210×160 RGB frames are processed by `AtariPreprocessing`:
- Converted to **grayscale** (removes irrelevant colour information).
- **Then they are downsampled to 84×84** pixels (reduces input dimensionality).
- **Frame-skipped by 4** (the chosen action is repeated for 4 frames; intermediate frames
  are max-pooled to handle flickering sprites — a known Atari hardware artefact).
- **`FrameStackObservation(env, 4)`** stacks the last 4 processed frames into a single
  observation of shape `(4, 84, 84)`, giving the agent temporal information about motion
  direction and velocity that a single frame cannot provide.


In [ ]:
# Environment creation 
SEED = 42

env = gym.make(
    "ALE/SpaceInvaders-v5",
    frameskip=1,                    # Raw env: no built-in frame skip
    repeat_action_probability=0.0,  # Deterministic actions (no sticky actions)
    full_action_space=False,        # Use minimal action set
    render_mode=None                # Headless for training
)

# Apply standard Atari preprocessing (grayscale, 84x84 resize, frame skip=4)
env = AtariPreprocessing(env, frame_skip=4)
# Stack 4 consecutive frames so the agent can perceive motion
env = FrameStackObservation(env, 4)

env.reset(seed=SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
env.action_space.seed(SEED)

print("=== Observation Space ===")
print(env.observation_space)
print()
print("=== Action Space ===")
print(env.action_space)
print()
print("=== Available Actions (Meanings) ===")

# Action set for Space Invaders
action_meanings = env.unwrapped.get_action_meanings()
for i, name in enumerate(action_meanings):
    print(f"  Action {i}: {name}")

=== Observation Space ===
Box(0, 255, (4, 84, 84), uint8)

=== Action Space ===
Discrete(6)

=== Available Actions (Meanings) ===
  Action 0: NOOP
  Action 1: FIRE
  Action 2: RIGHT
  Action 3: LEFT
  Action 4: RIGHTFIRE
  Action 5: LEFTFIRE


### Inputs from the Gym Environment

After preprocessing, each observation is a NumPy array of shape **(4, 84, 84)**:
- **4**: the number of stacked frames (temporal window).
- **84 × 84**: the spatial resolution of each grayscale frame.
- Pixel values are integers in **[0, 255]** (rescaled to [0, 1] inside the network).

The environment returns a **tuple** `(observation, reward, terminated, truncated, info)` on
each `step()` call:
- `observation`: the stacked frame array described above.
- `reward`: the change in game score (clipped to [-1, 1] during training for stability).
- `terminated`: `True` if the game episode has ended (all lives lost).
- `truncated`: `True` if the episode was cut short by a time limit.
- `info`: auxiliary metadata (e.g., remaining lives).

### Joystick Control Settings (Action Space)

The **minimal action set** for Space Invaders contains **6 discrete actions**, corresponding to
the directions and fire combinations available on the original Atari joystick:

| Action ID | Meaning        |
|-----------|----------------|
| 0         | NOOP (do nothing) |
| 1         | FIRE           |
| 2         | RIGHT          |
| 3         | LEFT           |
| 4         | RIGHTFIRE      |
| 5         | LEFTFIRE       |

The agent selects one of these 6 actions at each decision step (every 4 raw frames due to
frame skipping). The action is then repeated for those 4 frames.


---
## 3. Vanilla DQN Implementation

### 3a. Data Capture and Pre-processing

Raw Atari frames are high-dimensional and redundant. The
preprocessing pipeline (applied by `AtariPreprocessing`) reduces this to a tractable input:

1. **Grayscale conversion**: Colour adds no useful information in Space Invaders, so collapsing the 3 colour
   channels to 1 reduces computation by 3×.
2. **Resize to 84×84**: Chosen to
   retain enough spatial detail while keeping the CNN input manageable.
3. **Frame skipping (k=4)**: Actions are repeated for 4 frames, and the maximum pixel value
   across the last 2 frames is taken (max-pooling). This handles sprite flickering and speeds
   up training (the agent makes decisions at ~15 Hz instead of ~60 Hz).
4. **Frame stacking (n=4)**: Using 4 frames instead of 1 lets the agent infer motion, such as the velocity and direction of aliens and bullets.
5. **Pixel normalisation**: Inside the network, pixel values are rescaled from [0, 255] to
   [0.0, 1.0] via a `Rescaling(1/255)` layer, improving gradient stability.
6. **Reward clipping**: Rewards are clipped to {-1, 0, +1} during training. This normalises
   the reward scale across different Atari games and prevents large reward values from
   dominating gradient updates.

### 3b. Network Structure

The DQN uses a **Convolutional Neural Network (CNN)** to map a stack of 4 preprocessed
frames directly to Q-values for each possible action — without hand-crafted features.

Architecture (following Mnih et al., 2015):

| Layer | Type | Output shape | Notes |
|-------|------|-------------|-------|
| Input | Lambda (transpose) | (84, 84, 4) | Reorders axes from (4,84,84) → (84,84,4) for Conv2D |
| 1 | Rescaling(1/255) | (84, 84, 4) | Normalise pixels to [0,1] |
| 2 | Conv2D(32, 8×8, stride 4, ReLU) | (20, 20, 32) | Detects large-scale spatial patterns |
| 3 | Conv2D(64, 4×4, stride 2, ReLU) | (9, 9, 64) | Mid-level feature extraction |
| 4 | Conv2D(64, 3×3, stride 1, ReLU) | (7, 7, 64) | Fine-grained feature extraction |
| 5 | Flatten | (3136,) | Vectorise feature maps |
| 6 | Dense(512, ReLU) | (512,) | High-level representation |
| 7 | Dense(num_actions, linear) | (6,) | One Q-value per action (no activation) |

The output layer uses **linear (no) activation** because Q-values are unbounded real numbers,
not probabilities. The action with the highest Q-value is selected greedily.

**Two identical networks are maintained**:
- **Online model** (`model`): updated every 4 steps via gradient descent.
- **Target model** (`model_target`): a frozen copy of the online model, updated every
  10,000 steps. Using a separate target network stabilises training by providing fixed
  regression targets during weight updates — a key innovation of DQN over naïve Q-learning
  with neural networks (Mnih et al., 2015).

### 3c. Q-Learning Update

DQN minimises the **Bellman residual** (temporal difference error) using gradient descent.

For a sampled transition (s, a, r, s', done) from the replay buffer, the target Q-value is:

> **y = r + γ · max_{a'} Q_target(s', a') · (1 − done)**

Where:
- r is the observed reward.
- γ = 0.99 is the discount factor (future rewards are worth 99% of immediate ones).
- `Q_target(s', a')` is the next-state Q-value from the **frozen target network**.
- `(1 − done)` zeroes the future-reward term at terminal states.

The online network is then updated to minimise the **Huber loss** between the predicted
Q-value `Q(s, a)` and the target `y`:

> **L = HuberLoss(y, Q(s, a))**

The Huber loss is less sensitive to outliers than MSE: it behaves like L2 for small errors
and L1 for large errors, which prevents exploding gradients caused by occasional very large
TD errors early in training.

**Experience Replay** (Lin, 1992): Transitions are stored in a circular replay buffer of
capacity 100,000. Mini-batches of 32 are sampled uniformly at random. This breaks the
temporal correlations between consecutive transitions that would otherwise destabilise
online learning, and allows each transition to be reused multiple times.

**ε-greedy Exploration**: The agent selects a random action with probability ε, otherwise it
selects `argmax Q(s, a)`. The first 10,000 frames use purely random actions to fill the replay buffer before any
learning begins. ε is then annealed linearly from 1.0 to 0.1 over the next 100,000 frames and
held at 0.1. The schedule is sized to the ~125,000-frame training budget used here, so the agent
spends most of the run exploiting what it has learned rather than acting randomly.


In [ ]:
# SECTION 3 – VANILLA DQN

# Hyperparameters
SEED               = 42
GAMMA              = 0.99          # Discount factor for future rewards
EPSILON_START      = 1.0           # Initial exploration rate
EPSILON_MIN        = 0.1           # Minimum exploration rate
EPSILON_RANDOM_FRAMES = 10_000     # Frames of pure random exploration before learning
EPSILON_GREEDY_FRAMES = 100_000    # Frames over which epsilon decays from 1.0 to 0.1 (fits the training budget)
EPSILON_DECAY      = (EPSILON_START - EPSILON_MIN) / EPSILON_GREEDY_FRAMES
BATCH_SIZE         = 32            # Mini-batch size for replay sampling
MAX_MEMORY         = 100_000       # Replay buffer capacity
UPDATE_AFTER       = 4            # Update online model every N steps
UPDATE_TARGET      = 10_000        # Sync target network every N steps
MAX_STEPS_EP       = 10_000        # Max steps per episode
MAX_EPISODES       = 250          # Stop after this many episodes (for demo; increase for real training)
LEARNING_RATE      = 0.00025       # Adam learning rate (Mnih et al., 2015)

In [5]:
# Environment
def make_env(seed=SEED):
    """Create and wrap the Space Invaders environment."""
    env = gym.make(
        "ALE/SpaceInvaders-v5",
        frameskip=1,
        repeat_action_probability=0.0,
        full_action_space=False,
        render_mode=None
    )
    env = AtariPreprocessing(env, frame_skip=4)   # Grayscale, 84x84, frame-skip=4
    env = FrameStackObservation(env, 4)            # Stack 4 frames
    env.reset(seed=seed)
    return env

env = make_env()
NUM_ACTIONS = env.action_space.n
print(f"Number of actions: {NUM_ACTIONS}")

Number of actions: 6


In [ ]:
# ── Network Definition ───────────────────────────────────────────────────
def create_q_model(num_actions):
    """
    Build the DQN CNN as described in Mnih et al. (2015).
    Input:  (batch, 4, 84, 84)  – 4 stacked grayscale frames
    Output: (batch, num_actions) – one Q-value per action
    """
    return keras.Sequential([
        keras.Input(shape=(4, 84, 84)),
        # Transpose from (4,84,84) → (84,84,4) for Conv2D (channels-last)
        layers.Lambda(
            lambda t: keras.ops.transpose(t, [0, 2, 3, 1]),
            output_shape=(84, 84, 4),
        ),
        layers.Rescaling(1.0 / 255.0),               # Normalise pixels to [0, 1]
        layers.Conv2D(32, 8, strides=4, activation="relu"),  # Large receptive field
        layers.Conv2D(64, 4, strides=2, activation="relu"),  # Mid-level features
        layers.Conv2D(64, 3, strides=1, activation="relu"),  # Fine-grained features
        layers.Flatten(),
        layers.Dense(512, activation="relu"),         # Fully-connected representation
        layers.Dense(int(num_actions), activation="linear"),  # Q-values (unbounded)
    ])

# Online network: trained continuously
model = create_q_model(NUM_ACTIONS)
# Target network: frozen copy, updated periodically for stable targets
model_target = create_q_model(NUM_ACTIONS)
model_target.set_weights(model.get_weights())  # Initialise identically

model.summary()


In [ ]:
# ── Optimizer and Loss ───────────────────────────────────────────────────
optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=1.0)
# Huber loss: L2 for small errors, L1 for large – robust to outlier TD errors
loss_function = keras.losses.Huber()

# ── Replay Buffer (circular, using deque for O(1) append/pop) ────────────
action_history       = collections.deque(maxlen=MAX_MEMORY)
state_history        = collections.deque(maxlen=MAX_MEMORY)
state_next_history   = collections.deque(maxlen=MAX_MEMORY)
rewards_history      = collections.deque(maxlen=MAX_MEMORY)
done_history         = collections.deque(maxlen=MAX_MEMORY)

# ── Tracking ─────────────────────────────────────────────────────────────
episode_reward_history = []  # Reward per episode (for plotting)
running_reward         = 0
episode_count          = 0
frame_count            = 0
epsilon                = EPSILON_START  # Start fully random

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# VANILLA DQN TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════════
print("Starting Vanilla DQN training...")
start_time = time.time()

while True:
    observation, _ = env.reset()
    state = np.array(observation)
    episode_reward = 0

    for timestep in range(1, MAX_STEPS_EP):
        frame_count += 1

        # ── Action Selection (ε-greedy) ───────────────────────────────────
        if frame_count < EPSILON_RANDOM_FRAMES or epsilon > np.random.rand():
            # Random action: pure exploration
            action = env.action_space.sample()
        else:
            # Greedy action: exploit learned Q-values
            state_tensor = keras.ops.convert_to_tensor(state[None])  # Add batch dim
            q_values = model(state_tensor, training=False)
            action = int(keras.ops.argmax(q_values[0]))

        # ── Epsilon Decay (linear annealing after random phase) ───────────
        if frame_count > EPSILON_RANDOM_FRAMES:
            epsilon = max(EPSILON_MIN, epsilon - EPSILON_DECAY)

        # ── Environment Step ──────────────────────────────────────────────
        state_next, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated      # ends the episode
        state_next = np.array(state_next)
        episode_reward += reward

        # Clip reward to {-1, 0, +1} for training stability
        clipped_reward = np.clip(reward, -1, 1)

        # ── Store Transition in Replay Buffer ─────────────────────────────
        action_history.append(action)
        state_history.append(state)
        state_next_history.append(state_next)
        done_history.append(float(terminated))  # only a true game over stops bootstrapping, not a time-limit cut
        rewards_history.append(clipped_reward)
        state = state_next

        # ── Learning Step (every UPDATE_AFTER frames, once buffer is warm) ─
        if frame_count % UPDATE_AFTER == 0 and frame_count > EPSILON_RANDOM_FRAMES and len(done_history) >= BATCH_SIZE:
            # Sample a random mini-batch from the replay buffer
            indices = np.random.randint(0, len(done_history), size=BATCH_SIZE)

            state_sample      = np.array([state_history[i]      for i in indices])
            state_next_sample = np.array([state_next_history[i] for i in indices])
            rewards_sample    = np.array([rewards_history[i]    for i in indices])
            action_sample     = np.array([action_history[i]     for i in indices])
            done_sample       = np.array([done_history[i]       for i in indices])

            # ── Compute Bellman Targets using TARGET network ──────────────
            # Q_target(s', a') for all a' — fixed weights prevent oscillation
            # Direct call instead of .predict(): much lower per-step overhead inside a training loop
            future_q = model_target(state_next_sample, training=False).numpy()
            # y = r + γ * max_a' Q_target(s', a') * (1 - done)
            updated_q = rewards_sample + GAMMA * np.max(future_q, axis=1) * (1 - done_sample)

            # ── Gradient Update on ONLINE network ────────────────────────
            masks = keras.ops.one_hot(action_sample, NUM_ACTIONS)  # Select taken action
            with tf.GradientTape() as tape:
                q_values = model(state_sample)
                # Extract Q-value for the action that was actually taken
                q_action = keras.ops.sum(q_values * masks, axis=1)
                loss = loss_function(updated_q, q_action)

            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))

        # Sync Target Network 
        if frame_count % UPDATE_TARGET == 0:
            model_target.set_weights(model.get_weights())
            elapsed = time.time() - start_time
            print(f"[Frame {frame_count:>8}] ep={episode_count:>4} | "
                  f"running_reward={running_reward:.2f} | ε={epsilon:.4f} | "
                  f"elapsed={elapsed/60:.1f}min")

        if done:
            break

    # Episode Bookkeeping 
    episode_reward_history.append(episode_reward)
    running_reward = np.mean(episode_reward_history[-100:])  # 100-episode moving avg
    episode_count += 1

    print(f"  Episode {episode_count:>4} | reward={episode_reward:.0f} | "
          f"running_avg={running_reward:.2f} | ε={epsilon:.4f} | frames={frame_count}")

    # Stopping Conditions 
    if running_reward > 5000:
        print(f"\nSolved at episode {episode_count}!")
        break
    if MAX_EPISODES > 0 and episode_count >= MAX_EPISODES:
        print(f"\nStopped at episode {episode_count} (MAX_EPISODES reached).")
        break

print("Training complete.")

---
## 4. Results and Evaluation – Vanilla DQN

### 4b. How Do We Evaluate an RL Agent?

Evaluating an RL agent differs from supervised learning: there is no held-out test set.
Standard practice is to report:

1. **Episode reward**: The total (undiscounted) reward accumulated in one episode. Higher
   is better; for Space Invaders, this equals the game score.
2. **Running average reward** (over the last 100 episodes): Smooths out episode-to-episode
   variance and reveals the learning trend. The Atari benchmark considers a game "solved"
   if the agent exceeds a human-normalised threshold.
3. **ε over time**: Confirms that exploration is being reduced as expected.
4. **Loss curve** (if logged): Decreasing Huber loss indicates that the Q-value estimates
   are converging.

### 4c. Is the Agent Learning?

Each agent is trained for 250 episodes (roughly 125,000 frames). Because ε reaches its floor of 0.1 after
110,000 frames, the later episodes mostly reflect the learned policy rather than random play. Signs of learning are:
- Episode rewards trending upward after the ε-random phase.
- Running average reward increasing and stabilising.
- ε decaying from 1.0 → 0.1 by ~1M frames.

The plots below show the reward trajectory for the current run.


In [ ]:
import matplotlib.pyplot as plt

# ── Plot 1: Episode Reward ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw episode rewards
axes[0].plot(episode_reward_history, alpha=0.6, color="steelblue", label="Episode Reward")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Total Reward (Game Score)")
axes[0].set_title("Vanilla DQN – Episode Reward (Space Invaders)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Running average
window = min(10, len(episode_reward_history))
running_avg = np.convolve(episode_reward_history,
                          np.ones(window)/window, mode='valid')
axes[1].plot(running_avg, color="orange", linewidth=2,
             label=f"{window}-Episode Moving Average")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Average Reward")
axes[1].set_title("Vanilla DQN – Running Average Reward")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("vanilla_dqn_rewards.png", dpi=150)
plt.show()
print(f"Max episode reward: {max(episode_reward_history):.0f}")
print(f"Final running average ({window}-ep): {running_avg[-1]:.2f}")

In [ ]:
# ── Plot 2: Epsilon Decay Visualisation ──────────────────────────────────
frames_axis = np.arange(0, frame_count + 1)
epsilon_curve = np.maximum(
    EPSILON_MIN,
    EPSILON_START - EPSILON_DECAY * np.maximum(0, frames_axis - EPSILON_RANDOM_FRAMES)
)

plt.figure(figsize=(8, 4))
plt.plot(frames_axis, epsilon_curve, color="crimson", linewidth=2)
plt.axvline(x=EPSILON_RANDOM_FRAMES, linestyle="--", color="gray", label="End random phase")
plt.xlabel("Frame Count")
plt.ylabel("Epsilon (ε)")
plt.title("Exploration Rate Decay over Training Frames")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("epsilon_decay.png", dpi=150)
plt.show()
print(f"Current epsilon: {epsilon:.4f}")
print(f"Frames completed: {frame_count:,}")

---
## 5. Extension: Double DQN (DDQN)
### 5a. Problem Being Addressed
Vanilla DQN suffers from **overestimation bias**, where the algorithm overestimates the value of actions. Vanilla DQN calculates the values of actions by employing one network to choose the best course of action and estimate its value. In this scenario, a problem arises if the network chooses a wrong policy since no other network would tell what is wrong with it, and the result would be overestimation of action values and, hence, poor strategies. (Thrun & Schwartz, 1993; van Hasselt, 2010).
Van Hasselt et al. (2016) showed this overestimation happens across many Atari games and stops the agent from learning the best possible strategy.

### 5b. Theoretical Solution
DDQN solves this issue by separating the two responsibilities across two separate neural networks:

Action Selection: The online network selects the optimal action for the next state
Action Evaluation: The target network evaluates how valuable the selected action is

Instead of having one network agreeing with itself, two independent neural networks cross-verify each other. As they have distinct weightings and make distinct errors, it is very unlikely that they will both make the same error at the same time. This significantly minimizes the issue of overestimation.
This is the only difference from Vanilla DQN – everything else about the neural network architecture, experience replay buffer, and training process remains the same.

**References:**
- Van Hasselt, H., Guez, A., & Silver, D. (2016). Deep reinforcement learning with double Q-learning. *AAAI*, 30(1).
- Van Hasselt, H. (2010). Double Q-learning. *NeurIPS*, 23.
- Thrun, S., & Schwartz, A. (1993). Issues in using function approximation for reinforcement learning. *Proc. Connectionist Models Summer School*.


In [ ]:
# SECTION 5 – DOUBLE DQN IMPLEMENTATION

# Reset environment and create fresh models for DDQN
env_ddqn = make_env(seed=SEED + 1)

# Online and target networks (same architecture as Vanilla DQN)
model_ddqn        = create_q_model(NUM_ACTIONS)
model_target_ddqn = create_q_model(NUM_ACTIONS)
model_target_ddqn.set_weights(model_ddqn.get_weights())

optimizer_ddqn = keras.optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=1.0)
loss_fn_ddqn   = keras.losses.Huber()

# Fresh replay buffer for DDQN
action_buf   = collections.deque(maxlen=MAX_MEMORY)
state_buf    = collections.deque(maxlen=MAX_MEMORY)
next_buf     = collections.deque(maxlen=MAX_MEMORY)
reward_buf   = collections.deque(maxlen=MAX_MEMORY)
done_buf     = collections.deque(maxlen=MAX_MEMORY)

ddqn_reward_history = []
running_reward_ddqn = 0
ep_count_ddqn       = 0
frame_count_ddqn    = 0
epsilon_ddqn        = EPSILON_START

print("Starting Double DQN training...")
start_ddqn = time.time()

while True:
    observation, _ = env_ddqn.reset()
    state = np.array(observation)
    episode_reward = 0

    for timestep in range(1, MAX_STEPS_EP):
        frame_count_ddqn += 1

        # Action Selection (ε-greedy) (exploration vs exploitation)
        if frame_count_ddqn < EPSILON_RANDOM_FRAMES or epsilon_ddqn > np.random.rand():
            action = env_ddqn.action_space.sample()
        else:
            state_tensor = keras.ops.convert_to_tensor(state[None])
            q_vals = model_ddqn(state_tensor, training=False)
            action = int(keras.ops.argmax(q_vals[0]))

        # Epsilon Decay 
        if frame_count_ddqn > EPSILON_RANDOM_FRAMES:
            epsilon_ddqn = max(EPSILON_MIN, epsilon_ddqn - EPSILON_DECAY)

        # Environment Step 
        next_obs, reward, terminated, truncated, _ = env_ddqn.step(action)
        done = terminated or truncated
        next_state = np.array(next_obs)
        episode_reward += reward

        # Clip reward to {-1, 0, +1}
        clipped_reward = np.clip(reward, -1, 1)

        # Store Transition 
        action_buf.append(action)
        state_buf.append(state)
        next_buf.append(next_state)
        done_buf.append(float(terminated))  # only a true game over stops bootstrapping
        reward_buf.append(clipped_reward)
        state = next_state

        # DDQN Learning Step 
        if frame_count_ddqn % UPDATE_AFTER == 0 and frame_count_ddqn > EPSILON_RANDOM_FRAMES and len(done_buf) >= BATCH_SIZE:
            indices = np.random.randint(0, len(done_buf), size=BATCH_SIZE)

            s_sample  = np.array([state_buf[i]  for i in indices])
            sn_sample = np.array([next_buf[i]   for i in indices])
            r_sample  = np.array([reward_buf[i] for i in indices])
            a_sample  = np.array([action_buf[i] for i in indices])
            d_sample  = np.array([done_buf[i]   for i in indices])

            # KEY DDQN CHANGE 
            # Step 1: ONLINE network selects the best next action
            next_q_online  = model_ddqn(sn_sample, training=False).numpy()
            best_actions   = np.argmax(next_q_online, axis=1)  # argmax by ONLINE net

            # Step 2: TARGET network evaluates that action (decoupled!)
            next_q_target  = model_target_ddqn(sn_sample, training=False).numpy()
            best_next_q    = next_q_target[np.arange(BATCH_SIZE), best_actions]

            # Bellman target with DDQN correction
            updated_q = r_sample + GAMMA * best_next_q * (1 - d_sample)

            # Gradient Update 
            masks = keras.ops.one_hot(a_sample, NUM_ACTIONS)
            with tf.GradientTape() as tape:
                q_values = model_ddqn(s_sample)
                q_action = keras.ops.sum(q_values * masks, axis=1)
                loss = loss_fn_ddqn(updated_q, q_action)

            grads = tape.gradient(loss, model_ddqn.trainable_variables)
            optimizer_ddqn.apply_gradients(zip(grads, model_ddqn.trainable_variables))

        # Sync Target Network 
        if frame_count_ddqn % UPDATE_TARGET == 0:
            model_target_ddqn.set_weights(model_ddqn.get_weights())
            elapsed = time.time() - start_ddqn
            print(f"[Frame {frame_count_ddqn:>8}] ep={ep_count_ddqn:>4} | "
                  f"running_reward={running_reward_ddqn:.2f} | ε={epsilon_ddqn:.4f} | "
                  f"elapsed={elapsed/60:.1f}min")

        if done:
            break

    ddqn_reward_history.append(episode_reward)
    running_reward_ddqn = np.mean(ddqn_reward_history[-100:])
    ep_count_ddqn += 1

    print(f"  Episode {ep_count_ddqn:>4} | reward={episode_reward:.0f} | "
          f"running_avg={running_reward_ddqn:.2f} | ε={epsilon_ddqn:.4f} | "
          f"frames={frame_count_ddqn}")

    if running_reward_ddqn > 5000:
        print(f"\nSolved at episode {ep_count_ddqn}!")
        break
    if MAX_EPISODES > 0 and ep_count_ddqn >= MAX_EPISODES:
        print(f"\nStopped at episode {ep_count_ddqn} (MAX_EPISODES reached).")
        break


print("DDQN training complete.")

### 5d. Results and Evaluation – Double DQN

In [ ]:
# Plot: DDQN Reward History 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ddqn_reward_history, alpha=0.6, color="mediumseagreen", label="DDQN Episode Reward")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Total Reward")
axes[0].set_title("Double DQN – Episode Reward (Space Invaders)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

window_d = min(10, len(ddqn_reward_history))
ddqn_avg = np.convolve(ddqn_reward_history, np.ones(window_d)/window_d, mode='valid')
axes[1].plot(ddqn_avg, color="darkgreen", linewidth=2,
             label=f"{window_d}-Episode Moving Average")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Average Reward")
axes[1].set_title("Double DQN – Running Average Reward")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("ddqn_rewards.png", dpi=150)
plt.show()
print(f"DDQN Max episode reward: {max(ddqn_reward_history):.0f}")

### 5e. Comparison: Vanilla DQN vs Double DQN

The core algorithmic difference between Vanilla DQN and Double DQN is in a single line
of the target computation:

| | Target Q-value |
|---|---|
| **Vanilla DQN** | r + γ · max_{a'} **Q_target**(s', a') |
| **Double DQN** | r + γ · **Q_target**(s', **argmax_{a'} Q_online**(s', a')) |

In Vanilla DQN, the same network (`model_target`) both selects and evaluates the next
action. This coupling allows overestimation errors to compound: if the target network
overestimates Q(s', a*) for some action a*, that inflated value is directly used as the
regression target, pushing the online network to overestimate Q(s, a) correspondingly.

In Double DQN, the online network's noisy Q-values select the action, but the target
network — with *different* parameters — evaluates it. Since the two networks' estimation
errors are largely uncorrelated, the probability that both simultaneously overestimate
the same action is much lower. Van Hasselt et al. (2016) demonstrated that DDQN reduces
the overestimation by a factor of 5–10× on several Atari games and achieves higher scores
in 41 out of 49 Atari games compared to Vanilla DQN.

In a full-length training run (≥2M frames), we would expect DDQN to show:
- **More stable learning curves** (less variance in the running average).
- **Higher final performance** on Space Invaders.
- **Faster convergence** to high-scoring policies.

With only ~125,000 frames per agent, the gap between the two methods may be small or within
run-to-run noise; a fair comparison would need several seeds and a much longer run.


In [ ]:
# ── Side-by-side comparison plot ─────────────────────────────────────────
min_len = min(len(episode_reward_history), len(ddqn_reward_history))

plt.figure(figsize=(10, 5))
plt.plot(episode_reward_history[:min_len], alpha=0.5, color="steelblue",  label="Vanilla DQN")
plt.plot(ddqn_reward_history[:min_len],   alpha=0.5, color="mediumseagreen", label="Double DQN")

# Overlay smoothed curves
window_c = min(5, min_len)
v_smooth = np.convolve(episode_reward_history[:min_len], np.ones(window_c)/window_c, mode='valid')
d_smooth = np.convolve(ddqn_reward_history[:min_len],   np.ones(window_c)/window_c, mode='valid')

plt.plot(v_smooth, color="navy",      linewidth=2, label=f"Vanilla DQN {window_c}-ep avg")
plt.plot(d_smooth, color="darkgreen", linewidth=2, label=f"DDQN {window_c}-ep avg")

plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("Vanilla DQN vs Double DQN – Space Invaders")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("comparison.png", dpi=150)
plt.show()

### 5f. Gameplay of the Trained Agent

In [ ]:
# ── Record one greedy episode of the better agent and show it inline ─────
# Requires: pip install imageio imageio-ffmpeg
from IPython.display import HTML
from base64 import b64encode
import imageio

# Pick whichever agent scored higher over its last 50 training episodes
dqn_last  = np.mean(episode_reward_history[-50:])
ddqn_last = np.mean(ddqn_reward_history[-50:])
best_model, best_name = (model_ddqn, "Double DQN") if ddqn_last >= dqn_last else (model, "Vanilla DQN")
print(f"Recording {best_name} (last-50 avg: DQN={dqn_last:.1f}, DDQN={ddqn_last:.1f})")

# Same settings and wrappers as training, but rendering RGB frames
env_record = gym.make("ALE/SpaceInvaders-v5", frameskip=1, repeat_action_probability=0.0,
                      full_action_space=False, render_mode="rgb_array")
env_record = AtariPreprocessing(env_record, frame_skip=4)
env_record = FrameStackObservation(env_record, 4)

EVAL_EPSILON = 0.05   # small randomness so the agent can't get stuck repeating one action
frames, total_reward = [], 0
obs, _ = env_record.reset(seed=SEED)
state = np.array(obs)
for _ in range(MAX_STEPS_EP):
    frames.append(env_record.render())
    if np.random.rand() < EVAL_EPSILON:
        action = env_record.action_space.sample()
    else:
        q_values = best_model(state[None], training=False)
        action = int(keras.ops.argmax(q_values[0]))
    obs, reward, terminated, truncated, _ = env_record.step(action)
    total_reward += reward
    state = np.array(obs)
    if terminated or truncated:
        break
env_record.close()
print(f"Episode score: {total_reward:.0f}")

imageio.mimsave("gameplay.mp4", frames, fps=30, macro_block_size=1)
mp4 = open("gameplay.mp4", "rb").read()
HTML(f'<video width=400 controls><source src="data:video/mp4;base64,{b64encode(mp4).decode()}" type="video/mp4"></video>')


---
## 6. Reflective Piece

This assignment gave me hands-on experience of an end-to-end deep reinforcement learning pipeline. The key points I learned:

**The gap between theory and practice.** Understanding the theory was not enough; applying it in practice took a deeper understanding of the Bellman equation and ε-greedy exploration, the importance of the target network update interval, the need for reward clipping to avoid exploding gradients, and how sensitive training is to hyperparameters such as the learning rate and replay buffer size.

**The significance of the replay buffer and target network.** Without experience replay, consecutive transitions are highly correlated, which makes gradient descent unstable. Without a target network, the regression targets move with every update, causing oscillation. Understanding *why* these stabilising techniques exist, rather than only *how* to implement them, gave me real insight into the original DQN paper.

**Matching the exploration schedule to the compute budget.** My first run annealed ε over 1,000,000 frames but only trained for about 125,000, so ε never fell below 0.88 and the agent was still acting almost randomly. Sizing the schedule to the frames actually available is as important as the algorithm itself.

**Computational constraints.** I ran out of compute units several times during training, which cost time restarting runs. Calling the networks directly instead of `model.predict()` inside the loop removed a large per-step overhead.

**The elegance of Double DQN.** DDQN needs only one change to the Vanilla DQN target calculation, yet it fixes a real flaw in the original. It showed me how pinpointing the exact source of an error can lead to a simple, effective solution.


---
## 7. Level of Difficulty

**Self-assessed score: 3 / 3**

This submission demonstrates learning across multiple levels:

Putting the theory into practice was challenging and required a deeper understanding of the material.
1. **Understanding of concepts**: Realising the instability of DQN without a target network,
   the importance of replay buffer, and mathematics of the overestimation bias
   in Vanilla DQN and its solution using Double DQN.
2. **Correctness of implementation**: Implementation of both Vanilla DQN and Double DQN is fully completed
   starting from the template code, incorporating all important parts including target network update, linear
   epsilon decay, reward clipping, and computing the correct Bellman target value.
3. **Extension beyond the template**: The DDQN target computation is a non-trivial addition
   requiring understanding of the algorithm, not merely following a tutorial.
4. **Evaluation methodology**: Reward plotting, moving averages, and epsilon decay
   visualisation demonstrate awareness of how RL agents are evaluated.


---
## 8. Statement of Work

**Adesayo Akintunde ID: 21316651**:
Responsible for the full implementation including environment setup, Vanilla DQN training
loop, Double DQN extension, all text descriptions, plotting code, and reflective writing.




---
## 9. Use of Generative AI

The following prompts were submitted to Claude (Anthropic) during the preparation of this
assignment:

Explain this assignment to me. Help me understand q learning/ Help me understand DQN and DDQN


---
## 10. References

1. Mnih, V., Kavukcuoglu, K., Silver, D., Rusu, A. A., Veness, J., Bellemare, M. G., ... & Hassabis, D. (2015). Human-level control through deep reinforcement learning. *Nature*, 518(7540), 529–533. https://doi.org/10.1038/nature14236

2. Van Hasselt, H., Guez, A., & Silver, D. (2016). Deep reinforcement learning with double Q-learning. *Proceedings of the AAAI Conference on Artificial Intelligence*, 30(1). https://arxiv.org/abs/1509.06461

3. Van Hasselt, H. (2010). Double Q-learning. *Advances in Neural Information Processing Systems*, 23. https://papers.nips.cc/paper/2010/hash/091d584fced301b442654dd8c23b3fc9-Abstract.html

4. Lin, L.-J. (1992). Self-improving reactive agents based on reinforcement learning, planning and teaching. *Machine Learning*, 8(3–4), 293–321. https://doi.org/10.1007/BF00992699

5. Farama Foundation. (2023). Gymnasium documentation. https://gymnasium.farama.org/


6. Keras Team. (2023). Deep Q-Network Breakout. https://keras.io/examples/rl/deep_q_network_breakout/
